# env_tools

> MCP tools for conda/mamba environment management

In [ ]:
#| default_exp tools.env

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, Optional

import sys
from pathlib import Path

from rich.panel import Panel
from mcp.server.fastmcp import FastMCP

from nbdev_mcp.utils.paths import expand, discover_env_name, env_file, resolve_selector
from nbdev_mcp.utils.subprocess import which, run
from nbdev_mcp.utils.rich import make_console, get_output, render_result

## Environment Tools

In [ ]:
#| export
def ensure_env(
    project: Optional[str] = None,
    update: bool = False
) -> Dict[str, Any]:
    """Create or update the project's conda environment from its env YAML file.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    update : bool
        If True, update existing env; otherwise create new.
    
    Returns
    -------
    Dict[str, Any]
        Result with command output and status.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    env_name = discover_env_name(p)
    ef = env_file(p)
    exe = which(['mamba', 'conda'])
    
    c = make_console()
    if not ef.exists() or not env_name:
        c.print(Panel(
            "No env.<lib>.yml found or missing 'name:' field in YAML.",
            title='Environment'
        ))
        return {'ok': False, 'pretty': get_output(c)}
    
    if not exe:
        c.print(Panel(
            "Neither 'mamba' nor 'conda' was found on PATH.",
            title='Environment'
        ))
        return {'ok': False, 'pretty': get_output(c)}
    
    cmd = [exe, 'env', 'update' if update else 'create', '-f', str(ef)]
    if update and env_name:
        cmd += ['-n', env_name]
    
    logs = run(cmd, cwd=p)
    meta = {'env_file': str(ef), 'env_name': env_name, 'project': str(p)}
    pretty = render_result('Environment created/updated', meta, logs)
    return {**logs, **meta, 'pretty': pretty}

In [ ]:
#| export
def export_env(
    project: Optional[str] = None,
    out_path: Optional[str] = None
) -> Dict[str, Any]:
    """Export the current environment to the project's env YAML file.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    out_path : str, optional
        Custom output path for the YAML file.
    
    Returns
    -------
    Dict[str, Any]
        Result with exported file path and command output.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    ef = env_file(p) if out_path is None else expand(out_path)
    exe = which(['mamba', 'conda']) or 'conda'
    
    cmd = [exe, 'env', 'export', '--no-builds', '--prefix', sys.prefix, '--file', str(ef)]
    logs = run(cmd, cwd=p)
    
    meta = {'exported_to': str(ef), 'project': str(p)}
    pretty = render_result('Environment exported', meta, logs)
    return {**logs, **meta, 'pretty': pretty}

## MCP Registration

In [ ]:
#| export
def add_env_tools(mcp: FastMCP) -> None:
    """Attach environment management tools to the MCP server.
    
    Parameters
    ----------
    mcp : FastMCP
        The MCP server instance.
    """
    mcp.add_tool(ensure_env)
    mcp.add_tool(export_env)

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()